# Prompt selection

After running all the prompt versions we now select an ideal prompt of our 5 versions

## Selection 

We select the prompt with the best mean F1 across all three providers. We note an abnormal refusal to answer or error rate since they will be excluded from the F1 score. We care equally about both FN and FP cases, we therefore use the macro F1. The argument which of the two cases matter more to a geocoding verifier can be made separately, for consistency purposes we account for both here.

In [48]:
import hashlib
from pathlib import Path

import duckdb
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from thesis.config import load_config
from thesis.vlm.vlm_adapter import VLMResponse

prompt_lookup = {hashlib.sha256(p["text"].encode()).hexdigest(): p["id"] for p in load_config("prompts")["prompts"]}
ablation_dir = ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")

In [49]:
responses = []
for provider_dir in (
    Path("/Users/phillip/repositories/bachelor-thesis/data/vlm_cache") / p for p in ("openai", "anthropic", "google")
):  # noqa: E501
    for f in provider_dir.glob("*.json"):
        r = VLMResponse.model_validate_json(f.read_text())
        responses.append(
            {
                "provider": r.provider,
                "image_sha256": r.image_sha256,
                "prompt_id": prompt_lookup.get(r.prompt_sha256),
                "parsed_answer": r.parsed_answer,
                "refusal_error_state": r.refusal_error_state,
            }
        )
responses_df = pd.DataFrame(responses)

con = duckdb.connect(str(ablation_dir / "manifest.duckdb"), read_only=True)
input_samples = con.sql(
    "SELECT * FROM samples WHERE marker_shape = ? AND marker_color = ?",
    params=["crosshair", str((230, 20, 20))],
).df()
input_samples["image_sha256"] = input_samples["image_path"].apply(
    lambda p: hashlib.sha256((ablation_dir / str(p)).read_bytes()).hexdigest()
)


input_samples["image_sha256"].duplicated().sum()

np.int64(5)

In [50]:
dupes = input_samples[input_samples["image_sha256"].duplicated(keep=False)].sort_values("image_sha256")
dupes[["sample_id", "image_sha256", "target_signed_edge_distance_m", "ground_truth_inside"]]

,sample_id,image_sha256,target_signed_edge_distance_m,ground_truth_inside
448,9af73ba2a195f34058084ad44df768d0e594cf954b0ac6...,144c2fb7c46d15fc51d22f41aa1682e5b1c9c72d8f0ffc...,-5.0,True
458,2ce2faff32b518777317dedf32f614a537255e17d5403d...,144c2fb7c46d15fc51d22f41aa1682e5b1c9c72d8f0ffc...,NaN,True
0,ee8390b3eb091275b064d31c9c4efbcf5f9200bb2c8761...,274fd2f389dd957827e0b0f61ef4a4c32de807aaa44713...,-5.0,True
9,bbcaa86341daa40ef2a3614591b794ed367f83bdd82d09...,274fd2f389dd957827e0b0f61ef4a4c32de807aaa44713...,NaN,True
322,3892487d514b0e68a6ef895db6c4ae7ad7d313a1ffe48f...,c9afbc4ea7c204d0a979d10ca2761c0ab22ed8a0390490...,-5.0,True
332,c40a354970b577ce651bf8ec383c927a9ad5368fe0ab31...,c9afbc4ea7c204d0a979d10ca2761c0ab22ed8a0390490...,NaN,True
837,3b7dfd592ebc3570039499d5c4e4df9ae547f5eae7b5bd...,e5b1aa0d28dc0301a9971596f5ae46ee9e038c471bf8e8...,-5.0,True
847,06550c742e193cb4a89087548a884b5a8e7d01d12dea75...,e5b1aa0d28dc0301a9971596f5ae46ee9e038c471bf8e8...,NaN,True
942,9b31a125a9d659f0b352756ab42f9c97615ee41cbf5df5...,f23236e060083f1a436a562084870580ce5d7f5221b1d0...,-5.0,True
952,dfdf3f70499f92cd83f19d2cd450b93c5998f40daed07a...,f23236e060083f1a436a562084870580ce5d7f5221b1d0...,NaN,True


We found 5 hash collision, meaning images were the same. It shows that this is the case for -5m offset images vs control images. This means that -5m was the same as not shifting the centered pin. No mismatch or labeling or quality issue. Fine 

In [51]:
clean_samples = input_samples.sort_values("include_in_distance_curve", ascending=False).drop_duplicates(
    "image_sha256", keep="first"
)

merged = pd.merge(responses_df, clean_samples, on="image_sha256", validate="many_to_one")

In [52]:
merged["answered"] = merged["parsed_answer"].isin(["yes", "no"])
merged["predicted_positive"] = merged["parsed_answer"] == "yes"

rows = []
for (provider, prompt_id), group in merged.groupby(["provider", "prompt_id"]):
    answered = group[group["answered"]]
    precision, recall, f1, _ = precision_recall_fscore_support(
        answered["ground_truth_inside"], answered["predicted_positive"], average="macro", zero_division=0
    )
    rows.append(
        {
            "provider": provider,
            "prompt_id": prompt_id,
            "response_rate": group["answered"].mean(),
            "yes_rate": group["predicted_positive"].mean(),
            "accuracy": accuracy_score(answered["ground_truth_inside"], answered["predicted_positive"]),
            "precision_macro": precision,
            "recall_macro": recall,
            "f1_macro": f1,
            "n": len(group),
            "n_answered": len(answered),
        }
    )
summary = pd.DataFrame(rows)

In [53]:
summary

,provider,prompt_id,response_rate,yes_rate,accuracy,precision_macro,recall_macro,f1_macro,n,n_answered
0,anthropic,details,0.698113,0.180735,0.706970,0.670024,0.645086,0.651249,1007,703
1,anthropic,minimal,0.898709,0.285005,0.633149,0.583029,0.581002,0.581844,1007,905
2,anthropic,roleplaying,0.934459,0.176763,0.691817,0.635513,0.594717,0.596694,1007,941
3,anthropic,some_details,0.934459,0.379345,0.595112,0.566377,0.571880,0.565926,1007,941
4,anthropic,super_detailed_and_role,1.000000,0.229394,0.691162,0.639995,0.611981,0.616634,1007,1007
5,google,details,1.000000,0.452830,0.819265,0.805921,0.842992,0.810293,1007,1007
6,google,minimal,1.000000,0.467726,0.810328,0.800587,0.838621,0.802211,1007,1007
7,google,roleplaying,1.000000,0.394240,0.850050,0.828734,0.855241,0.837679,1007,1007
8,google,some_details,1.000000,0.453823,0.814300,0.801140,0.837758,0.805169,1007,1007
9,google,super_detailed_and_role,1.000000,0.345581,0.799404,0.772829,0.779199,0.775758,1007,1007


In [59]:
avg_macro_f1 = summary.groupby("prompt_id")["f1_macro"].mean()
refusal_rates_provider = summary.groupby("provider")["response_rate"].mean()
refusal_rates_prompt = summary.groupby("prompt_id")["response_rate"].mean()

print("Average macro F1 by provider:")
print(avg_macro_f1)
print("\nAverage refusal rate by provider:")
print(refusal_rates_provider)
print("\nAverage refusal rate by prompt:")
print(refusal_rates_prompt)

Average macro F1 by provider:
prompt_id
details                    0.693331
minimal                    0.656604
roleplaying                0.702050
some_details               0.666741
super_detailed_and_role    0.688544
Name: f1_macro, dtype: float64

Average refusal rate by provider:
provider
anthropic    0.893148
google       1.000000
openai       1.000000
Name: response_rate, dtype: float64

Average refusal rate by prompt:
prompt_id
details                    0.899371
minimal                    0.966236
roleplaying                0.978153
some_details               0.978153
super_detailed_and_role    1.000000
Name: response_rate, dtype: float64


We select the `roleplaying` prompt to use for our study as the ideal prompt. We note that the `response_rate` of Anthropic for this prompt lies at `0.934459`

- anthropic specifically shows a ideal F1 at the details level. This is however inflated by the high refusal rate. will be investigated separately. Our criterion for selection claims roleplaying as ideal prompt without account for exclusion of that value. 
- each provider slightly diverges. Google ideal is coincidentally chose, however openai has near best F1 for the same chosen prompt. Anthropic operates at a slight disadvantage. The shared prompt is not ideal per provider and was never meant to be. 
- Refusal only by Anthropic 
- TODO:
    - check if anthropic refusals cluster on hard cases, low pin shift
    - This result changes how each stages F1 needs to be thought about
    - Precision: The rounding needs to be capped to two digits, 1k samples